In [15]:
import time
import re
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup


# JobPosting class to store job information
class JobPosting:
    def __init__(self, title, location, salary, date_posted, closing_date, contract_type, working_pattern, job_link):
        self.reference_number = None
        self.title = title
        self.location = location
        self.working_pattern = working_pattern
        self.salary = salary
        self.date_posted = date_posted
        self.closing_date = closing_date
        self.contract_type = contract_type
        self.grade = None
        self.duration = None
        self.job_link = job_link
        self.job_summary = None
        self.main_duties = None
        self.job_description = None
        self.team_structure = None
        self.qualifications = None
        self.employer_name = None
        self.employer_contact = None
        self.employer_address = None
        self.disclosure_check = None
        self.certificate_of_sponsorship = None
        self.uk_registration = None
        self.pay_scheme = None

    def __repr__(self):
        return (
            f"Reference Number: {self.reference_number}\n\n"
            f"Job Title: {self.title}\n\n"
            f"Location: {self.location}\n\n"
            f"Working Pattern: {self.working_pattern}\n\n"
            f"Salary: {self.salary}\n\n"
            f"Date Posted: {self.date_posted}\n\n"
            f"Closing Date: {self.closing_date}\n\n"
            f"Contract Type: {self.contract_type}\n\n"
            f"Grade: {self.grade}\n\n"
            f"Duration: {self.duration}\n\n"
            f"Job Link: {self.job_link}\n\n"
            f"Job Summary: {self.job_summary}\n\n"
            f"Main Duties: {self.main_duties}\n\n"
            f"Job Description: {self.job_description}\n\n"
            f"Team Structure: {self.team_structure}\n\n"
            f"Qualifications: {self.qualifications}\n\n"
            f"Employer Name: {self.employer_name}\n\n"
            f"Employer Contact: {self.employer_contact}\n\n"
            f"Employer Address: {self.employer_address}\n\n"
            f"Disclosure Check: {self.disclosure_check}\n\n"
            f"Certificate of Sponsorship: {self.certificate_of_sponsorship}\n\n"
            f"UK Registration: {self.uk_registration}\n\n"
            f"Pay Scheme: {self.pay_scheme}\n\n"
        )


# Helper functions
def clean_text(text):
    return ' '.join(text.split()).strip() if text else 'N/A'

def clean_list(text):
    if not text:
        return 'N/A'
    text = re.sub(r'\b(?:e\.g|i\.e|etc|Dr|Mr|Mrs|Ms|Jr|Sr|vs|Inc|Ltd|Co|Prof|PhD|M\.D|B\.Sc)\.', 
                  lambda m: m.group(0).replace('.', '[DOT]'), text)
    return '\n'.join(f"- {sentence.strip()}." for sentence in text.split('.') if sentence.strip()).replace('[DOT]', '.')

def clean_text_with_strong_tags(text, tag):
    soup = BeautifulSoup(text, 'html.parser')
    for strong_tag in soup.find_all(tag):
        strong_tag.string = f"**{strong_tag.text}**"
    return ' '.join(soup.stripped_strings)

def extract_tag_text(soup, tag_name, partial_string, next_tag_stop='h2', nested_tag='p'):
    tag = soup.find(tag_name, string=lambda text: text and partial_string.lower() in text.lower())
    if tag:
        content = [clean_text_with_strong_tags(str(sibling), 'strong') for sibling in tag.find_next_siblings()
                   if sibling.name == nested_tag and sibling.name != next_tag_stop]
        return clean_text(' '.join(content)) if content else 'N/A'
    return 'N/A'

def extract_qualifications(soup, tag_name, partial_string, next_tag_stop='h2'):
    tag = soup.find(tag_name, string=lambda text: text and partial_string.lower() in text.lower())
    if tag:
        qualifications = [clean_text(li.text) for sibling in tag.find_next_siblings()
                          if sibling.name == 'ul' for li in sibling.find_all('li') if sibling.name != next_tag_stop]
        return ' '.join(qualifications) if qualifications else 'N/A'
    return 'N/A'


# Updated async function to extract detailed job information
async def extract_job_details(page, job_posting):
    try:
        await page.goto(job_posting.job_link, wait_until='networkidle')
        await page.wait_for_selector('main.nhsuk-main-wrapper')
        job_detail_html = await page.content()
        detail_soup = BeautifulSoup(job_detail_html, 'html.parser')
        main_content = detail_soup.find('main', class_='nhsuk-main-wrapper')
        
        if main_content:
            job_posting.job_summary = clean_list(extract_tag_text(main_content, 'h3', 'summary'))
            job_posting.main_duties = clean_list(extract_tag_text(main_content, 'h3', 'duties'))
            job_posting.team_structure = clean_list(extract_tag_text(main_content, 'h3', 'about us'))
            job_posting.qualifications = clean_list(extract_qualifications(main_content, 'h2', 'specification'))
            job_posting.job_description = clean_list(extract_tag_text(main_content, 'h2', 'description'))
    
            job_posting.working_pattern = extract_tag_text(main_content, 'h3', 'working pattern')
            employer_name_tag = main_content.find('p', id='employer_name_details')
            job_posting.employer_name = clean_text(employer_name_tag.text) if employer_name_tag else 'N/A'
    
            address_fields = ['employer_address_line_1_a', 'employer_address_line_2_b', 'employer_town_c', 'employer_postcode_e']
            job_posting.employer_address = clean_text(' '.join([clean_text(main_content.find('p', id=field).text or '')
                                                                for field in address_fields if main_content.find('p', id=field)])) or 'N/A'
    
            employer_contact_tag = main_content.find('p', id='employer_website_url')
            if employer_contact_tag:
                employer_contact_link = employer_contact_tag.find('a', id='employer_website_url_link')
                job_posting.employer_contact = clean_text(employer_contact_link['href']) if employer_contact_link else 'N/A'
            else:
                job_posting.employer_contact = 'N/A'
    
            job_posting.disclosure_check = clean_text(main_content.find('div', id='dbs-container').text if main_content.find('div', id='dbs-container') else 'N/A')
            job_posting.certificate_of_sponsorship = clean_text(main_content.find('h3', id='tier-two-sponsorship').find_next('p').text if main_content.find('h3', id='tier-two-sponsorship') else 'N/A')
            job_posting.uk_registration = clean_text(main_content.find('h3', id='uk-registration').find_next('p').text if main_content.find('h3', id='uk-registration') else 'N/A')
            job_posting.pay_scheme = clean_text(main_content.find('p', id='payscheme-type').text if main_content.find('p', id='payscheme-type') else 'None')
            job_posting.grade = clean_text(main_content.find('p', id='payscheme-band').text if main_content.find('p', id='payscheme-band') else 'None')
            job_posting.reference_number = clean_text(main_content.find('p', id='trac-job-reference').text if main_content.find('p', id='trac-job-reference') else 'None')
            job_posting.duration = clean_text(main_content.find('p', id='contract_duration').text if main_content.find('p', id='contract_duration') else 'None')

    except Exception as e:
        print(f"Error extracting details for {job_posting.title}: {e}")
        
    
async def scrape_jobs_playwright(url):
    job_listings = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        page_number = 1
        
        try:
            while True:
                await page.goto(url)
                await page.wait_for_selector('.nhsuk-list.search-results')
                soup = BeautifulSoup(await page.content(), 'html.parser')
                job_elements = soup.find_all('li', class_='nhsuk-list-panel')

                for job in job_elements:
                    title_tag = job.find('a', {'data-test': 'search-result-job-title'})
                    title = clean_text(title_tag.text) if title_tag else 'N/A'
                    job_link = f"https://www.jobs.nhs.uk{title_tag['href']}" if title_tag else 'N/A'
                    location = clean_text(job.find('div', {'data-test': 'search-result-location'}).text or 'N/A')
                    salary = clean_text(job.find('li', {'data-test': 'search-result-salary'}).find('strong').text or 'N/A')
                    date_posted = clean_text(job.find('li', {'data-test': 'search-result-publicationDate'}).find('strong').text or 'N/A')
                    closing_date = clean_text(job.find('li', {'data-test': 'search-result-closingDate'}).find('strong').text or 'N/A')
                    contract_type = clean_text(job.find('li', {'data-test': 'search-result-jobType'}).find('strong').text or 'N/A')
                    working_pattern = clean_text(job.find('li', {'data-test': 'search-result-workingPattern'}).find('strong').text or 'N/A')

                    job_posting = JobPosting(title, location, salary, date_posted, closing_date, contract_type, working_pattern, job_link)
                    await extract_job_details(page, job_posting)
                    job_listings.append(job_posting)

                print(f"Page {page_number} scraped")

                # Check if there is a next page (based on the new HTML structure)
                pagination = soup.find('nav', class_='recordset-pager')
                if pagination:
                    next_page_tag = pagination.find('a', class_='zcicon icon-only ic-pager-next', href=True)
                    if next_page_tag:
                        next_page_link = f"https://www.healthjobsuk.com{next_page_tag['href']}"
                        print(f"Next page found: {next_page_link}")  # Debug message to confirm next page link is found
                        url = next_page_link
                        page_number += 1
                    else:
                        print("No next page link found.")  # Debug message if no next page is found
                        break
                else:
                    print("No pagination found.")  # Debug message if pagination is missing
                    break

        except Exception as e:
            print(f"Error while scraping page {page_number}: {e}")
        finally:
            await browser.close()

    return job_listings


async def main():
    base_url = "https://www.jobs.nhs.uk/candidate/search/results?"
    keyword = "FY2, CT1, CT2, ST1, ST2, ST3, LAS, Trust doctor, Trust grade"
    pay_band = "SPECIALTY_DOCTOR,FOUNDATION_DOCTOR,DOCTOR_OTHER"
    pay_range = "30-40,40-50"
    sort_by = "publicationDateDesc"
    language = "en"

    url = (f"{base_url}keyword={keyword.replace(' ', '%20')}&payBand={pay_band}&payRange={pay_range}"
           f"&skipPhraseSuggester=true&searchFormType=sortBy&sort={sort_by}&language={language}")

    start_time = time.time()

    # Directly await the scraping function without running asyncio.run()
    jobs = await scrape_jobs_playwright(url)

    print(f"Scraping completed in {time.time() - start_time:.2f} seconds")

    for job in jobs:
        print(job")
        print("-" * 50)

# To execute the code, just call the main() function inside an async loop in Jupyter
await main()


SyntaxError: unterminated string literal (detected at line 215) (3045745286.py, line 215)

In [17]:
import time
import hashlib
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup


# JobPosting class to store job information
class JobPosting:
    def __init__(self, title, grade, employer_name, location, speciality, salary, job_link):
        self.title = title
        self.grade = grade
        self.employer_name = employer_name
        self.location = location
        self.speciality = speciality  # This can map to job_summary or main_duties in NHS if needed
        self.salary = salary
        self.job_link = job_link
        # New fields aligned with the NHS structure
        self.contract_type = None
        self.working_pattern = None
        self.reference_number = None
        self.closing_date = None
        self.date_posted = None  # Optional, specific to NHS
        self.duration = None  # Optional, specific to NHS
        self.job_summary = None  # Optional, specific to NHS
        self.main_duties = None  # Optional, specific to NHS
        self.job_description = None
        self.team_structure = None
        self.qualifications = None
        self.employer_contact = None
        self.employer_address = None  # Optional, specific to NHS
        self.disclosure_check = None  # Optional, specific to NHS
        self.certificate_of_sponsorship = None  # Optional, specific to NHS
        self.uk_registration = None  # Optional, specific to NHS
        self.pay_scheme = None  # Optional, specific to NHS

    def __repr__(self):
        return (
            f"Job Title: {self.title}\n"
            f"Grade: {self.grade}\n"
            f"Employer: {self.employer_name}\n"
            f"Location: {self.location}\n"
            f"Speciality: {self.speciality}\n"
            f"Salary: {self.salary}\n"
            f"Contract Type: {self.contract_type}\n"
            f"Working Pattern: {self.working_pattern}\n"
            f"Reference Number: {self.reference_number}\n"
            f"Closing Date: {self.closing_date}\n"
            f"Date Posted: {self.date_posted}\n"
            f"Duration: {self.duration}\n"
            f"Job Summary: {self.job_summary}\n"
            f"Main Duties: {self.main_duties}\n"
            f"Job Description: {self.job_description}\n"
            f"Team Structure: {self.team_structure}\n"
            f"Qualifications: {self.qualifications}\n"
            f"Employer Contact: {self.employer_contact}\n"
            f"Employer Address: {self.employer_address}\n"
            f"Disclosure Check: {self.disclosure_check}\n"
            f"Certificate of Sponsorship: {self.certificate_of_sponsorship}\n"
            f"UK Registration: {self.uk_registration}\n"
            f"Pay Scheme: {self.pay_scheme}\n"
            f"Job Link: {self.job_link}\n"
        )


# Helper function to clean text
def clean_text(text):
    return ' '.join(text.split()).strip() if text else 'N/A'


# Function to generate a unique ID if reference_number doesn't exist
def generate_unique_id(job_posting):
    # Concatenate the fields to generate a unique identifier
    identifier_str = f"{job_posting.title}_{job_posting.employer_name}_{job_posting.location}"
    return hashlib.sha256(identifier_str.encode('utf-8')).hexdigest()


# Async function to extract detailed job information
async def extract_job_details(page, job_posting):
    try:
        await page.goto(job_posting.job_link, wait_until='networkidle')
        await page.wait_for_selector('main#hj-main')
        job_detail_html = await page.content()
        detail_soup = BeautifulSoup(job_detail_html, 'html.parser')

        # Extract speciality (Main area)
        speciality_tag = detail_soup.find('dt', string='Main area')
        if speciality_tag:
            job_posting.speciality = clean_text(speciality_tag.find_next('dd').text)

        # Extract contract type (Contract)
        contract_type_tag = detail_soup.find('dt', string='Contract')
        if contract_type_tag:
            job_posting.contract_type = clean_text(contract_type_tag.find_next('dd').text)

        # Extract working pattern (Hours)
        working_pattern_tag = detail_soup.find('dt', string='Hours')
        if working_pattern_tag:
            working_pattern_text = working_pattern_tag.find_next('dd').text
            job_posting.working_pattern = clean_text(working_pattern_text)

        # Extract reference number (Job ref)
        reference_number_tag = detail_soup.find('dt', string='Job ref')
        if reference_number_tag:
            job_posting.reference_number = clean_text(reference_number_tag.find_next('dd').text)
        else:
            # If no reference number exists, generate one using the unique ID function
            job_posting.reference_number = generate_unique_id(job_posting)

        # Extract salary (already present but override if necessary)
        salary_tag = detail_soup.find('dt', string='Salary')
        if salary_tag:
            job_posting.salary = clean_text(salary_tag.find_next('dd').text)

        # Extract closing date
        closing_date_tag = detail_soup.find('dt', string='Closing')
        if closing_date_tag:
            job_posting.closing_date = clean_text(closing_date_tag.find_next('dd').text)

        # Extract employer contact details (Name, Job title, Email address, Telephone number)
        employer_contact_tag = detail_soup.find('section', id='hj-enquiry-html')
        if employer_contact_tag:
            contact_name = clean_text(employer_contact_tag.find('dt', string='Name').find_next('dd').text) if employer_contact_tag.find('dt', string='Name') else 'N/A'
            contact_job_title = clean_text(employer_contact_tag.find('dt', string='Job title').find_next('dd').text) if employer_contact_tag.find('dt', string='Job title') else 'N/A'
            contact_email = clean_text(employer_contact_tag.find('dt', string='Email address').find_next('dd').text) if employer_contact_tag.find('dt', string='Email address') else 'N/A'
            contact_phone = clean_text(employer_contact_tag.find('dt', string='Telephone number').find_next('dd').text) if employer_contact_tag.find('dt', string='Telephone number') else 'N/A'
            job_posting.employer_contact = f"{contact_name}, {contact_job_title}, {contact_email}, {contact_phone}"
        else:
            job_posting.employer_contact = 'N/A'

        # Extract team structure (id="hj-employer-header")
        team_structure_tag = detail_soup.find('section', id='hj-employer-header')
        if team_structure_tag:
            job_posting.team_structure = clean_text(team_structure_tag.get_text(separator=' ', strip=True))

        # Extract job description (id="hj-job-advert")
        job_description_tag = detail_soup.find('section', id='hj-job-advert')
        if job_description_tag:
            job_posting.job_description = clean_text(job_description_tag.get_text(separator=' ', strip=True))

        # Extract qualifications (id="hj-job-role-requirement")
        qualifications_tag = detail_soup.find('section', id='hj-job-role-requirement')
        if qualifications_tag:
            job_posting.qualifications = clean_text(qualifications_tag.get_text(separator=' ', strip=True))

    except Exception as e:
        print(f"Error extracting details for {job_posting.title}: {e}")


# Async function to scrape the job listings
async def scrape_jobs_playwright(url):
    job_listings = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        page_number = 1
        try:
            while True:
                await page.goto(url)
                await page.wait_for_selector('main#hj-main')
                soup = BeautifulSoup(await page.content(), 'html.parser')

                job_elements = soup.find_all('li', class_='hj-job')
                for job in job_elements:
                    # Extracting job title
                    title_tag = job.find('div', class_='hj-jobtitle')
                    title = clean_text(title_tag.text) if title_tag else 'N/A'

                    # Extracting job link
                    job_link_tag = job.find('a', href=True)
                    job_link = f"https://www.healthjobsuk.com{job_link_tag['href']}" if job_link_tag else 'N/A'

                    # Extracting grade
                    grade_tag = job.find('div', class_='hj-grade')
                    grade = clean_text(grade_tag.text) if grade_tag else 'N/A'

                    # Extracting employer name
                    employer_name = clean_text(job.find('div', class_='hj-employername').text) if job.find('div', class_='hj-employername') else 'N/A'

                    # Extracting location
                    location = clean_text(job.find('div', class_='hj-locationtown').text) if job.find('div', class_='hj-locationtown') else 'N/A'

                    # Extracting speciality
                    speciality_tag = job.find('div', class_='hj-primaryspeciality')
                    speciality = clean_text(speciality_tag.text) if speciality_tag else 'N/A'

                    # Extracting salary
                    salary_tag = job.find('div', class_='hj-salary')
                    salary = clean_text(salary_tag.text) if salary_tag else 'N/A'

                    # Creating a JobPosting object
                    job_posting = JobPosting(title, grade, employer_name, location, speciality, salary, job_link)

                    # Extract detailed information from the job link
                    await extract_job_details(page, job_posting)

                    job_listings.append(job_posting)

                print(f"Page {page_number} scraped")

                # Check if there is a next page (based on the new HTML structure)
                pagination = soup.find('nav', class_='recordset-pager')
                if pagination:
                    next_page_tag = pagination.find('a', class_='page-link', href=True, title='Next page')
                    if next_page_tag:
                        next_page_link = f"https://www.healthjobsuk.com{next_page_tag['href']}"
                        # Wait for 2 seconds before moving to the next page
                        await asyncio.sleep(2)

                        # Update URL to scrape the next page
                        url = next_page_link
                        page_number += 1
                    else:
                        print("No next page link found.")  # Debug message if no next page is found
                        break

        except Exception as e:
            print(f"Error while scraping page {page_number}: {e}")
        finally:
            await browser.close()

    return job_listings


# Main function to run the scraping for multiple URLs
async def main():
    urls = [
        "https://www.healthjobsuk.com/job_list/ns?JobSearch_q=FY2&JobSearch_QueryIntegratedSubmit=Search&_tr=JobSearch&_ts=39274&_srt=startdate&_sd=a",
        "https://www.healthjobsuk.com/job_list?JobSearch_q=FY2&JobSearch_d=&JobSearch_g=&JobSearch_re=_POST&JobSearch_re_0=1&JobSearch_re_1=1-_-_-&JobSearch_re_2=1-_-_--_-_-&JobSearch_Submit=Search&_tr=JobSearch&_ts=28444&_srt=startdate&_sd=a"
    ]
    
    all_jobs = []  # Store all jobs from all URLs
    start_time = time.time()

    # Loop over each URL and scrape jobs
    for url in urls:
        print(f"Scraping URL: {url}")
        jobs = await scrape_jobs_playwright(url)
        all_jobs.extend(jobs)  # Add jobs from this URL to the overall list

    print(f"Scraping completed in {time.time() - start_time:.2f} seconds")

    # Output the jobs
    for job in all_jobs:
        print("A job scraped:")
        print(job)
        print("-" * 50)


# Call the main function inside an async loop
await main()


Page 1 scraped
Pagination found: <nav aria-label="Pagination" class="recordset-pager">
<ul class="pagination">
<li class="page-item active">
<a aria-current="page" aria-label="Page 1" class="page-link" data-toggle="tooltip" data-tooltip-aria-hidden="true" href="" title="Page 1: current page">
					1
				</a>
</li>
<li class="page-item">
<a aria-label="Page 2" class="page-link" data-toggle="tooltip" data-tooltip-aria-hidden="true" href="/job_list?JobSearch_q=&amp;JobSearch_d=&amp;JobSearch_g=&amp;JobSearch_re=_POST&amp;JobSearch_re_0=1&amp;JobSearch_re_1=1-_-_-&amp;JobSearch_re_2=1-_-_--_-_-&amp;JobSearch_Submit=Search&amp;_tr=JobSearch&amp;_ts=1&amp;_pg=2&amp;_pgid=" title="Page 2">
					2
				</a>
</li>
<li class="page-item">
<a aria-label="Page 3" class="page-link" data-toggle="tooltip" data-tooltip-aria-hidden="true" href="/job_list?JobSearch_q=&amp;JobSearch_d=&amp;JobSearch_g=&amp;JobSearch_re=_POST&amp;JobSearch_re_0=1&amp;JobSearch_re_1=1-_-_-&amp;JobSearch_re_2=1-_-_--_-_-&amp;

InvalidStateError: invalid state